#### From Text to Diagnosis:
#### Evaluating GPT-Based Models for Classifying Depression Severity in Online Texts

# Script 5 - NAIVE BAYES & LOGISTIC REGRESSION


In [ ]:
import pandas as pd
import re
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, mean_absolute_error, mean_squared_error, confusion_matrix, precision_score, recall_score, f1_score, classification_report
from imblearn.under_sampling import RandomUnderSampler
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer 
import nltk

nltk.download('stopwords')
nltk.download('wordnet')

### Load data

In [ ]:
# Load the data
training = 'C:/Users/45233/Downloads/data train.csv'
validation = 'C:/Users/45233/Downloads/data-validation.csv'
test = 'C:/Users/45233/Downloads/data test.csv'
train_set = pd.read_csv(training)
valid_set = pd.read_csv(validation)
test_set = pd.read_csv(test)


### Defining the text processing function
In the code, we define the function to perform lemmatization as well as remove capitalization, special symbols and stopwords.

In [ ]:
# Define text processing function
def process_text(text, remove_stopwords=False, remove_capitalization=False, remove_special_symbols=False, lemmatize=False):
    processed_text = text
    lemmatizer = WordNetLemmatizer()
    if remove_capitalization:
        processed_text = processed_text.lower()
    if remove_special_symbols:
        processed_text = re.sub(r'\W', ' ', processed_text)
    if remove_stopwords:
        stop_words = set(stopwords.words('english'))
        processed_text = ' '.join([word for word in processed_text.split() if word not in stop_words])
    if lemmatize:
        processed_text = ' '.join([lemmatizer.lemmatize(word) for word in processed_text.split()])
    return processed_text

In [ ]:
# Define the prepare_dataframe function
def prepare_dataframe(df):
    # Apply text transformations
    df['text_remove_stopwords'] = df['text'].apply(lambda x: process_text(x, remove_stopwords=True))
    df['text_remove_capitalization'] = df['text'].apply(lambda x: process_text(x, remove_capitalization=True))
    df['text_remove_special_symbols'] = df['text'].apply(lambda x: process_text(x, remove_special_symbols=True))
    df['text_all'] = df['text'].apply(lambda x: process_text(x, remove_stopwords=True, remove_capitalization=True, remove_special_symbols=True, lemmatize=True)) 

    # Combine with multi_label_number
    df_result = df[['multi_label_number', 'text_remove_stopwords', 'text_remove_capitalization', 'text_remove_special_symbols', 'text_all']]
    
    return df_result

In [ ]:
# Prepare the datasets
train_set_processed = prepare_dataframe(train_set)
valid_set_processed = prepare_dataframe(valid_set)
test_set_processed = prepare_dataframe(test_set)

In [ ]:
# Initialization of vectorizers.
vectorizers = {
    'BoW': CountVectorizer(),
    'TF-IDF': TfidfVectorizer()
}

# Define classifiers.
classifiers = {
    'Naive Bayes': MultinomialNB(),
    'Logistic Regression': LogisticRegression(max_iter=1000)
}

### Create a function to fit classifier and evaluate the output

In [ ]:

# Function to train and evaluate models.
def train_and_evaluate(vectorizer_name, vectorizer, classifier_name, classifier, X_train_resampled, y_train_resampled, X_valid, y_valid, X_test, y_test):
    print(f"\n{classifier_name} with {vectorizer_name} Results:")

    # Fit the classifier. Note that 'resampled' is in the name, because we will balance the training set as seen in the next code.
    classifier.fit(X_train_resampled, y_train_resampled)

    # Predict on the validation set.
    y_pred_valid = classifier.predict(X_valid)

    # Calculate metrics for the validation set.
    accuracy_valid = accuracy_score(y_valid, y_pred_valid)
    mae_valid = mean_absolute_error(y_valid, y_pred_valid)
    mse_valid = mean_squared_error(y_valid, y_pred_valid)
    conf_matrix_valid = confusion_matrix(y_valid, y_pred_valid)
    precision_valid = precision_score(y_valid, y_pred_valid, average='macro')
    recall_valid = recall_score(y_valid, y_pred_valid, average='macro')
    f1_valid = f1_score(y_valid, y_pred_valid, average='macro')

    print("Validation Set Metrics:")
    print("Accuracy Score:", accuracy_valid)
    print("Mean Absolute Error:", mae_valid)
    print("Mean Squared Error:", mse_valid)
    print("Confusion Matrix:\n", conf_matrix_valid)
    print("Precision Score:", precision_valid)
    print("Recall Score:", recall_valid)
    print("F1 Score:", f1_valid)
    print(classification_report(y_valid, y_pred_valid, target_names=['Class 0', 'Class 1', 'Class 2', 'Class 3']))

    # Predict on the test set.
    y_pred_test = classifier.predict(X_test)

    # Calculate metrics for the test set.
    accuracy_test = accuracy_score(y_test, y_pred_test)
    mae_test = mean_absolute_error(y_test, y_pred_test)
    mse_test = mean_squared_error(y_test, y_pred_test)
    conf_matrix_test = confusion_matrix(y_test, y_pred_test)
    precision_test = precision_score(y_test, y_pred_test, average='macro')
    recall_test = recall_score(y_test, y_pred_test, average='macro')
    f1_test = f1_score(y_test, y_pred_test, average='macro')

    print("Test Set Metrics:")
    print("Accuracy Score:", accuracy_test)
    print("Mean Absolute Error:", mae_test)
    print("Mean Squared Error:", mse_test)
    print("Confusion Matrix:\n", conf_matrix_test)
    print("Precision Score:", precision_test)
    print("Recall Score:", recall_test)
    print("F1 Score:", f1_test)
    print(classification_report(y_test, y_pred_test, target_names=['Class 0', 'Class 1', 'Class 2', 'Class 3']))



## Transforming the text data
Here we apply the vectorizer, which tokenizes the textual data. We also undersampled the majority class to balance the training set and initiatied the above function.

In [ ]:
# Loop through vectorizers and classifiers.
for vectorizer_name, vectorizer in vectorizers.items():
    # Fit and transform the training set on the 'text_all' column.
    X_train = vectorizer.fit_transform(train_set_processed['text_all'])
    y_train = train_set_processed['multi_label_number']

    # Balance the training set using RandomUnderSampler.
    rus = RandomUnderSampler(random_state=42)
    X_train_resampled, y_train_resampled = rus.fit_resample(X_train, y_train)

    # Transform the validation set and test set.
    X_valid = vectorizer.transform(valid_set_processed['text_all'])
    y_valid = valid_set_processed['multi_label_number']

    X_test = vectorizer.transform(test_set_processed['text_all'])
    y_test = test_set_processed['multi_label_number']

    # Loop through classifiers.
    for classifier_name, classifier in classifiers.items():
        train_and_evaluate(vectorizer_name, vectorizer, classifier_name, classifier, X_train_resampled, y_train_resampled, X_valid, y_valid, X_test, y_test)
